# Lesson 03 — SQLAlchemy ORM Models

**Run this in Google Colab** — every student gets the same environment.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mpalomera/learning-sql/blob/master/lessons/06-sqlalchemy-orm-migrations/03_sqlalchemy_models.ipynb)

---

## Step 1: Install dependencies

In [ ]:
!pip install sqlalchemy oracledb -q

In [ ]:
from google.colab import userdata

USERNAME = userdata.get('FREESQL_USER')
PASSWORD = userdata.get('FREESQL_PASSWORD')
DSN      = userdata.get('FREESQL_DSN')

print('credentials loaded')

## Step 2: Define ORM models

In [ ]:
from datetime import datetime
from sqlalchemy import (
    create_engine, Column, Integer, String,
    Text, ForeignKey, DateTime, CheckConstraint, func
)
from sqlalchemy.orm import declarative_base, relationship, Session

Base = declarative_base()

class Team(Base):
    __tablename__ = "teams"
    id          = Column(Integer, primary_key=True)
    name        = Column(String(50), nullable=False, unique=True)
    description = Column(String(200))
    created_at  = Column(DateTime, server_default=func.current_timestamp())
    users = relationship("User", back_populates="team")

    def __repr__(self):
        return f"<Team(id={self.id}, name='{self.name}')>"

class User(Base):
    __tablename__ = "users"
    id         = Column(Integer, primary_key=True)
    username   = Column(String(50), nullable=False, unique=True)
    email      = Column(String(100), nullable=False)
    full_name  = Column(String(100))
    team_id    = Column(Integer, ForeignKey("teams.id"))
    created_at = Column(DateTime, server_default=func.current_timestamp())
    team     = relationship("Team", back_populates="users")
    tasks    = relationship("Task", back_populates="assignee")
    comments = relationship("Comment", back_populates="user")

    def __repr__(self):
        return f"<User(id={self.id}, username='{self.username}')>"

class Task(Base):
    __tablename__ = "tasks"
    id           = Column(Integer, primary_key=True)
    title        = Column(String(200), nullable=False)
    description  = Column(String(1000))
    status       = Column(String(20), default="open")
    assigned_to  = Column(Integer, ForeignKey("users.id"))
    priority    = Column(String(10), default="medium")
    created_at   = Column(DateTime, server_default=func.current_timestamp())
    updated_at   = Column(DateTime, onupdate=func.current_timestamp())
    assignee = relationship("User", back_populates="tasks")
    comments = relationship("Comment", back_populates="task", cascade="all, delete-orphan")

    def __repr__(self):
        return f"<Task(id={self.id}, title='{self.title}', status='{self.status}')>"

class Comment(Base):
    __tablename__ = "comments"
    __table_args__ = (
        CheckConstraint("content != ''", name="ck_comments_content_nonempty"),
    )

    id         = Column(Integer, primary_key=True, autoincrement=True)
    task_id    = Column(Integer, ForeignKey("tasks.id", ondelete="CASCADE"), nullable=False)
    user_id    = Column(Integer, ForeignKey("users.id"), nullable=False)
    content    = Column(Text, nullable=False)
    created_at = Column(DateTime, default=datetime.utcnow)

    task = relationship("Task", back_populates="comments")
    user = relationship("User", back_populates="comments")

    def __repr__(self):
        return f"<Comment(id={self.id}, task_id={self.task_id}, user_id={self.user_id})>"

print("✅ Models defined: Team, User, Task, Comment")

## Step 3: Connect and query with ORM

In [ ]:
engine = create_engine(
    "oracle+oracledb://:@",
    connect_args={
        "user": USERNAME,
        "password": PASSWORD,
        "dsn": DSN
    }
)

with Session(engine) as session:
    print("🏢 Teams:")
    for team in session.query(Team).all():
        print(f"   {team}")
        for user in team.users:
            print(f"      -> {user.full_name} ({user.username})")

    print("\n📝 Tasks with assignees:")
    for task in session.query(Task).all():
        assignee = task.assignee.full_name if task.assignee else "Unassigned"
        print(f"   {task.title} → {assignee}")

print("\n✅ ORM models working! Relationships navigate automatically.")

# Lesson 03 — Alembic Migrations (Pure Python)


In [ ]:
!pip install sqlalchemy oracledb alembic -q

In [ ]:
import os
from alembic.config import Config
from alembic import command

# Create a minimal alembic.ini in memory
alembic_cfg = Config()
alembic_cfg.set_main_option('script_location', '/content/alembic')
alembic_cfg.set_main_option('sqlalchemy.url', 'oracle+oracledb://:@')

# Initialize the migration directory
!mkdir -p /content/alembic/versions

# Write a minimal env.py
env_py = '''
from logging.config import fileConfig
from sqlalchemy import engine_from_config
from alembic import context

# This is the Alembic Config object
config = context.config

# Add your model's MetaData object here for 'autogenerate' support
from __main__ import Base
target_metadata = Base.metadata

def run_migrations_offline():
    url = config.get_main_option('sqlalchemy.url')
    context.configure(url=url, target_metadata=target_metadata, literal_binds=True)
    with context.begin_transaction():
        context.run_migrations()

def run_migrations_online():
    connectable = engine_from_config(
        config.get_section(config.config_ini_section),
        prefix='sqlalchemy.',
        connect_args={'user': "''' + USERNAME + '''", 'password': "''' + PASSWORD + '''", 'dsn': "''' + DSN + '''"}
    )
    with connectable.connect() as connection:
        context.configure(connection=connection, target_metadata=target_metadata)
        with context.begin_transaction():
            context.run_migrations()

if context.is_offline_mode():
    run_migrations_offline()
else:
    run_migrations_online()
'''

with open('/content/alembic/env.py', 'w') as f:
    f.write(env_py)

# Write a minimal script.py.mako
script_template = '''"""${message}

Revision ID: ${up_revision}
Revises: ${down_revision | comma,n}
Create Date: ${create_date}
"""

from alembic import op
import sqlalchemy as sa
${imports if imports else ""}

# revision identifiers, used by Alembic.
revision = ${repr(up_revision)}
down_revision = ${repr(down_revision)}
branch_labels = ${repr(branch_labels)}
depends_on = ${repr(depends_on)}

def upgrade():
${upgrades if upgrades else "    pass"}

def downgrade():
${downgrades if downgrades else "    pass"}
'''

with open('/content/alembic/script.py.mako', 'w') as f:
    f.write(script_template)

print('✅ Alembic initialized in /content/alembic/')

In [ ]:
# Generate migration from models vs current database state
command.revision(alembic_cfg, autogenerate=True, message='Initial schema')

# Show what was generated
import glob
migration_files = sorted(glob.glob('/content/alembic/versions/*.py'))
print('Generated migrations:')
for f in migration_files:
    print(f'  {f}')

In [ ]:
# Read and display the latest migration
latest = migration_files[1]
with open(latest) as f:
    content = f.read()

print(content)


In [ ]:
team = Team()

In [ ]:

command.upgrade(alembic_cfg, 'head')
print('Migration applied! Tables created.')

In [ ]:
# Rollback one migration
command.downgrade(alembic_cfg, '-1')
print('Downgraded by 1. New columns removed.')

In [ ]:
import glob
import os

migration_files = glob.glob('/content/alembic/versions/*.py')

for f in migration_files:
    os.remove(f)
    print(f"Deleted: {f}")

# Exercise 2 — Migration Creation

In [ ]:
command.revision(
    alembic_cfg,
    autogenerate=True,
    message="add comments table"
)
print("Migration generated.")

In [ ]:
import glob

migration_files = sorted(
    glob.glob('/content/alembic/versions/*.py')
)

print("Migration files:")
for f in migration_files:
    print(f"  {f}")

latest = migration_files[-1]
print(f"\n--- Contents of {latest} ---\n")
with open(latest) as f:
    print(f.read())

In [ ]:
command.upgrade(alembic_cfg, 'head')
print("Migration applied!")

# Exercise 3 — CRUD Challenge

In [ ]:
with Session(engine) as session:
    devops_team = Team(name="DevOps", description="Infrastructure and operations team")
    session.add(devops_team)
    session.flush()
    print(f"Created: {devops_team}")

    diana = User(
        username="diana_ops",
        email="diana@example.com",
        full_name="Diana Ops",
        team_id=devops_team.id
    )
    session.add(diana)
    session.flush()
    print(f"Created: {diana}")

    PRIORITY_ORDER = {"low": 0, "medium": 1, "high": 2}

    task_high = Task(
        title="[HIGH] Deploy Kubernetes cluster",
        description="Set up production k8s cluster",
        priority="high",
        status="open",
        assigned_to=diana.id
    )
    task_med = Task(
        title="[MEDIUM] Configure CI/CD pipeline",
        description="Set up GitHub Actions workflows",
        priority="medium",
        status="open",
        assigned_to=diana.id
    )
    task_low = Task(
        title="[LOW] Update server documentation",
        description="Write runbook for server maintenance",
        priority="low",
        status="open",
        assigned_to=diana.id
    )
    session.add_all([task_high, task_med, task_low])
    session.flush()

    print(f"\nTask count for {diana.username}: {len(diana.tasks)}")
    for t in diana.tasks:
        print(f"   - {t.title} [{t.status}]")

    all_tasks = [task_high, task_med, task_low]
    highest = max(all_tasks, key=lambda t: PRIORITY_ORDER[t.priority])
    highest.status = "closed"
    print(f"\nClosed: {highest.title}")

    lowest = min(all_tasks, key=lambda t: PRIORITY_ORDER[t.priority])
    session.delete(lowest)
    print(f"Deleted: {lowest.title}")

    session.commit()

# Exercise 4 — Migration Rollback

In [ ]:
import glob, os

command.revision(alembic_cfg, message="add estimated_hours to tasks")

migration_files = sorted(glob.glob('/content/alembic/versions/*.py'))
latest = migration_files[-1]
print(f"Created migration: {latest}")

with open(latest) as f:
    content = f.read()

content = content.replace(
    "def upgrade():\n    pass",
    "def upgrade():\n    op.add_column('tasks', sa.Column('estimated_hours', sa.Integer(), nullable=True))"
)
content = content.replace(
    "def downgrade():\n    pass",
    "def downgrade():\n    op.drop_column('tasks', 'estimated_hours')"
)

with open(latest, 'w') as f:
    f.write(content)

command.upgrade(alembic_cfg, 'head')
print("Migration applied")

In [ ]:
command.downgrade(alembic_cfg, "-1")